# Lesson 24 — Generators: Practice (SUPPORTING)

Work in `lessons/lesson-24-generators/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. Readings are floats or numeric strings. Write each deliverable
in its stated file (R16 — no `solution.py`, no double-write), then run each
**Driver:** cell. Every `Expected` block must match byte-exact.

You create: `sensors.py` (E1–M2) · `alerts.py` + `harvest.py` (Challenge,
which imports `sensors.py` — write once, reuse).
Provided: `buggy_harvest.py` (buggy — true hunt, you fix it in place).
Installs: none — stdlib only.


## E1 — Rebuild batching as a generator function

## Intent
**Batch any stream into fixed-size windows with a five-line generator — the class twin.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `sensors.py`, define the generator function `batches(source, size)`.
Accumulate items; `yield` each full window; yield the ragged tail as a real
window; end by falling off (bare end, never `return` with a value). This is
the L23 `WindowReader` contract re-authored — same windows, no dunders.

**File tree**

- `sensors.py` (`batches`)

**Interface contract**

- `list(batches(source, size))` is the window list; empty source → zero
  windows; oversized `size` → one window.
- Calling `batches(...)` builds nothing (frozen until first pull).

**Constraints**

- **`size < 1` raises `ValueError("window size must be >= 1")` — on first
  pull, not at call** (a frozen body validates frozen-late; the driver
  consumes inside `try`).
- **The ragged tail is a real window, never dropped.**
- **No `return` with a value anywhere in the body.**

**Coverage**

- Generator function basics → lesson.md Core Concepts 1/6.

**Expected** (Driver cell):
```text
E1 OK
```


In [ ]:
import types
import sensors as s

assert list(s.batches([1, 2, 3, 4, 5], 2)) == [[1, 2], [3, 4], [5]]
assert list(s.batches([], 3)) == []
assert list(s.batches([1, 2, 3], 9)) == [[1, 2, 3]]
try:
    list(s.batches([1], 0))
except ValueError as e:
    assert str(e) == "window size must be >= 1"
else:
    raise AssertionError("no raise")
b = s.batches([1, 2], 2)
assert isinstance(b, types.GeneratorType)
assert next(b) == [1, 2]
print("E1 OK")

## E2 — One-line lazy transforms

## Intent
**Transform a stream with a generator expression — zero pulls at build.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `sensors.py`, define `over(source, threshold)` returning the generator
expression `(x for x in source if x > threshold)`. No `def`-with-`yield`
here — the whole deliverable is one frozen expression. Readings are
plain floats in this exercise.

**File tree**

- `sensors.py` (`over`)

**Interface contract**

- `over(source, threshold)` returns a generator yielding source items
  strictly over `threshold`, in order.

**Constraints**

- **The body is a single `return` of a genexp (no `yield` keyword).**
- **Building pulls nothing** (driver proves it with a counting source).

**Coverage**

- Generator expressions + pull clock → lesson.md Core Concepts 2–3.

**Expected** (Driver cell):
```text
E2 OK
```


In [ ]:
import importlib
import types
import sensors as s; importlib.reload(s)

g = s.over([18.5, 42.0, 19.0], 40.0)
assert isinstance(g, types.GeneratorType)
assert list(g) == [42.0]
pulled = []
def metered():
    for x in [18.5, 42.0]:
        pulled.append(x)
        yield x
m = s.over(metered(), 40.0)
assert pulled == []
assert next(m) == 42.0
assert pulled == [18.5, 42.0]
assert list(m) == []
print("E2 OK")

## E3 — Bound infinity, survive emptiness

## Intent
**Harvest a bounded prefix from an endless stream — and hold on empty input.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `sensors.py`, define `ticks(start=0)` (infinite `0, 1, 2, …` via
`while True` + `yield`) and `harvest(pred, stream, limit)` (first
`limit` items satisfying `pred`, via `for` + `break`; abandoned
generators simply left behind). Empty source yields `[]` first-pass —
the unannounced edge, stated here only as a requirement.

**File tree**

- `sensors.py` (`ticks`, `harvest`)

**Interface contract**

- `ticks(start)` counts up forever from `start`.
- `harvest(pred, stream, limit)` returns at most `limit` matching items.

**Constraints**

- **No new tools — `while`/`for`/`break` only.**
- **`limit == 0` returns `[]` without pulling once.**
- **Empty source returns `[]` (no crash).**

**Coverage**

- Infinity + bounding → lesson.md Core Concept 4.

**Expected** (Driver cell):
```text
E3 OK
```


In [ ]:
import importlib
import types
import sensors as s; importlib.reload(s)

t = s.ticks()
assert isinstance(t, types.GeneratorType)
assert [next(t) for _ in range(3)] == [0, 1, 2]
assert next(s.ticks(10)) == 10
assert s.harvest(lambda x: x % 10 == 0, s.ticks(), 3) == [0, 10, 20]
assert s.harvest(lambda x: True, iter([]), 5) == []
calls = []
def src():
    for x in [1, 2, 3, 4, 5]:
        calls.append(x)
        yield x
assert s.harvest(lambda x: True, src(), 0) == []
assert calls == []
print("E3 OK")

## M1 — A lazy validating pipeline stage

## Intent
**Parse a dirty reading stream into clean floats — pulling exactly what each pull needs.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `sensors.py`, define `parse_one(x: object) -> float` and the generator
`parse_valid(source)`. `parse_one` converts: `bool` → invalid; `int`/`float`
→ `float(x)`; `str` → stripped then `float()`; anything else → invalid.
`parse_valid` yields each converted value, skipping invalid lines
silently, pulling the source only as pulled itself (pull-1-advances-1).

**File tree**

- `sensors.py` (`parse_one`, `parse_valid`)

**Interface contract**

- `parse_one("  99.0 ") == 99.0`; exact messages below otherwise.
- `parse_valid(source)` yields clean floats; dry source yields nothing.

**Constraints**

- **`parse_one("")` (and blank-after-strip) raises
  `ValueError("invalid reading: empty")`; non-numeric / `None` / `bool`
  raises `ValueError("invalid reading: not a number")` — in that order.**
- **`parse_valid` is a generator (lazy stage, never a built list).**
- **One pull advances the source minimally** (driver counts pulls).

**Coverage**

- Lazy stage + boundary rule → lesson.md Core Concepts 3/5.

**Expected** (Driver cell):
```text
M1 OK
```


In [ ]:
import importlib
import types
import sensors as s; importlib.reload(s)

assert s.parse_one("  99.0 ") == 99.0
assert s.parse_one(40) == 40.0
for bad, msg in [("", "invalid reading: empty"),
                 ("   ", "invalid reading: empty"),
                 ("xx", "invalid reading: not a number"),
                 (None, "invalid reading: not a number"),
                 (True, "invalid reading: not a number")]:
    try:
        s.parse_one(bad)
    except ValueError as e:
        assert str(e) == msg, (bad, str(e))
    else:
        raise AssertionError(("no raise", bad))
g = s.parse_valid([21.5, "  99.0 ", "", "xx", None, True, 40])
assert isinstance(g, types.GeneratorType)
assert list(g) == [21.5, 99.0, 40.0]
calls = []
def src():
    for x in [10.0, "bad", 20.0]:
        calls.append(x)
        yield x
p = s.parse_valid(src())
assert next(p) == 10.0
assert calls == [10.0]
assert next(p) == 20.0
assert calls == [10.0, "bad", 20.0]
assert list(s.parse_valid([])) == []
print("M1 OK")

## M2 — Compose alert windows yourself

## Intent
**Build the hot-window pipeline from your own stages — order observable.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `sensors.py`, define the generator `alert_windows(source, size,
threshold)` from YOUR `parse_valid` + `batches` (this composition is the
exercise — the driver checks behavior only). Validate first so rot never
occupies window slots; batch the survivors; yield only windows holding
at least one reading strictly over `threshold`. Small driver data may
stay lazy end to end — no materialization needed, none permitted beyond
the current window.

**File tree**

- `sensors.py` (`alert_windows` — no new file)

**Constraints**

- **Validate-first: invalid lines never occupy window slots.**
- **Reuse your own stages (no reimplemented validation, no second
  batcher).**
- **Lazy: a generator in, windows out on pull (driver checks the type).**
- **`size` violations surface the E1 `ValueError` unchanged.**

**Coverage**

- Learner-authored composition → M2 law + Core Concepts 1/5.

**Expected** (Driver cell):
```text
M2 OK
```


In [ ]:
import importlib
import types
import sensors as s; importlib.reload(s)

a = s.alert_windows([18.0, "", 41.0, 19.0, 45.0, 20.0], 2, 40.0)
assert isinstance(a, types.GeneratorType)
assert list(a) == [[18.0, 41.0], [19.0, 45.0]]
assert list(s.alert_windows([18.0, 19.0], 2, 40.0)) == []
assert list(s.alert_windows(["", "  "], 2, 40.0)) == []
try:
    list(s.alert_windows([18.0], 0, 40.0))
except ValueError as e:
    assert str(e) == "window size must be >= 1"
else:
    raise AssertionError("no raise")
print("M2 OK")

## H1 — Hunt: four pipeline bugs in buggy_harvest

## Intent
**Diagnose four real generator failures in a provided module — fix each where it lives.**

**Type:** DEBUG — Hard — fixes 1 file

**Task**
`buggy_harvest.py` is provided, buggy: four planted failures, one per
Pitfall (exhausted reuse, return-instead-of-yield, eager link, shared
default). Nothing else in the file is wrong. Run the DEBUGGING protocol:
reproduce via the driver, read the traceback, locate, hypothesize, fix,
explain. Fix in place; do not restructure the module.

**File tree**

- `buggy_harvest.py` (`harvest_count`, `hot_lines`, `lazy_valid`,
  `collect` — YOU fix all four)

**Interface contract** (fixed states, exact)

- `harvest_count` returns `(total, items)` with a non-empty body after
  counting — on lists and one-shot iterators alike.
- `hot_lines` yields readings over threshold (never an empty stream
  from correct logic).
- `lazy_valid` pulls the source minimally (one pull advances one item).
- `collect` calls never share state.

**Constraints**

- **Four fixes only — no refactoring, no new symbols.**
- **Each fix stays in its own layer.**
- **Fill one 7-row chain per bug in the next cell, as you fix.**

**Coverage**

- The four generator failures → lesson.md Pitfalls 1–4.

**Expected** (Driver cell):
```text
H1 OK
```


### H1 chain reports (one per bug — fill as you fix)

#### Bug 1 — harvest_count
| Step | Content |
|---|---|
| Symptom | AssertionError assert (n, got) == (3, ["a", "b", "c"]) |
| Trigger | n, got = bh.harvest_count(["a", "b", "c"]) |
| Hypothesis | either wrong count or iterator issue |
| Evidence | 3, [] |
| Cause | iterator issue, re-accessing iter |
| Fix | for loop instead of using generator |
| Proof | 3 ['a', 'b', 'c'] |

#### Bug 2 — hot_lines
| Step | Content |
|---|---|
| Symptom | AssertionError assert list(bh.hot_lines([18.0, 45.0, 20.0], 40.0)) == [45.0] |
| Trigger | bh.hot_lines |
| Hypothesis | generator error |
| Evidence | [] |
| Cause | declaring comprehension list instead of generator |
| Fix | direct return the generator return (r for r in readings if r > threshold) |
| Proof | [45.0] |

#### Bug 3 — lazy_valid
| Step | Content |
|---|---|
| Symptom | AssertionError assert calls == [5], calls |
| Trigger | bh.lazy_valid |
| Hypothesis | fail on generator or validation |
| Evidence | [5, -1, 6] |
| Cause | set source as list: items = list(source) |
| Fix | delete it, direct use of source |
| Proof | [5] |

#### Bug 4 — collect
| Step | Content |
|---|---|
| Symptom | AssertionError assert bh.collect([9]) == [9] |
| Trigger | bh.collect |
| Hypothesis | reuse the same list |
| Evidence | [1, 2, 9] |
| Cause | acc=[] |
| Fix | set a fresh list  |
| Proof | H1 OK |


In [ ]:
import importlib
import types
import buggy_harvest as bh
importlib.reload(bh)

n, got = bh.harvest_count(["a", "b", "c"])
assert (n, got) == (3, ["a", "b", "c"])
n2, got2 = bh.harvest_count(iter(["a", "ERROR", "b"]))
assert (n2, got2) == (3, ["a", "ERROR", "b"])
assert isinstance(bh.hot_lines([18.0], 40.0), types.GeneratorType)
assert list(bh.hot_lines([18.0, 45.0, 20.0], 40.0)) == [45.0]
calls = []
def src():
    for x in [5, -1, 6]:
        calls.append(x)
        yield x
g = bh.lazy_valid(src())
assert next(g) == 5
assert calls == [5], calls
assert bh.collect([1, 2]) == [1, 2]
assert bh.collect([9]) == [9]
print("H1 OK")

## Challenge — Dawn harvest run over telemetry

## Intent
**Ship a harvest run over the sensor log: every hot reading reported, every rotten line skipped loudly — streamed end to end.**

**Type:** REAL-WORLD — Micro Build — writes 2 files

**Task**
The engineer runs one command before dawn:
`python harvest.py <telemetry.log> <alerts.txt>`. It streams the log
through windows, validates each line, collects readings over 40.0,
skips rotten lines with a loud `skip` line, and writes an alerts report
that is also printed — same bytes both places.

Create `alerts.py` and `harvest.py` in this folder. Reuse `sensors.py`
(`batches`, `parse_one` — write once, import, never double-write).

**Flow** (the whole wiring — every arrow is required)

```text
main reads <telemetry.log> as a streaming handle (never read_text for the log)
  → batches(enumerate(handle), 3): numbered lines in windows of 3
  → per (i, raw) in file order:
      value = parse_one(raw)               # rotten → skip line, no alert check
      value > 40.0 → alert line            # valid cool readings emit nothing
  → text = build_alerts(lines)
  → write text to <alerts.txt> AND print the same text to stdout
```

The log is always a valid text file — no corrupt-file handling required.
"Missing telemetry" means the path does not exist or is not a file.

**File tree**

- `alerts.py` (`build_alerts`)
- `harvest.py` (`main`)

**Interface contract**

- `alerts.build_alerts(lines)` returns header + lines + trailing
  newline (header alone when empty). It performs no I/O.
- Header shape: `alerts A, skipped S` (counts derived from line
  prefixes `alert ` / `skip `).
- `harvest.main(argv)` — house shape (L20–L23): full `sys.argv`
  (`main(sys.argv)` guard), paths at `argv[1]` / `argv[2]`, `-> None`,
  errors leave via `sys.exit(1)` after one stderr line.

**Constraints**

- **Consume the handle ONLY through `batches`** (never iterate it
  directly — laziness posture is review-graded).
- **Per line: parse → threshold-check → alert/skip line; valid cool
  readings emit nothing.**
- **Alert threshold is exactly `40.0` (strictly over).**
- **Wrong argc prints `usage: harvest.py <telemetry.log> <alerts.txt>`
  to stderr and exits 1.**
- **Missing telemetry prints `bad telemetry: not found` to stderr and
  exits 1.**
- **Stdout and the alerts file are byte-identical.**

**Coverage**

- Streaming + generators + boundary validation composed → lesson.md
  Core Concepts 1–6 + house entry (L20–L23 retrieval).

**Expected** (Driver cell, real subprocess runs, fixture of 8):
```text
CH OK
```


In [ ]:
import subprocess, sys, tempfile, pathlib
HOME = pathlib.Path.cwd()
log_lines = ["21.5", "41.0", "", "19.0", "xx", "45.5", "20.0", "39.9"]
expected = ("alerts 2, skipped 2\n" "alert 1: 41.0\n"
    "skip 2: invalid reading: empty\n"
    "skip 4: invalid reading: not a number\n" "alert 5: 45.5\n")
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    lg, rp = td / "telemetry.log", td / "alerts.txt"
    lg.write_text("\n".join(log_lines) + "\n", encoding="utf-8")
    run = subprocess.run(
        [sys.executable, "harvest.py", str(lg), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert run.returncode == 0, run.stderr
    assert run.stdout == expected, repr(run.stdout)
    assert rp.read_text(encoding="utf-8") == expected
    usage = subprocess.run(
        [sys.executable, "harvest.py", str(lg)],
        capture_output=True, text=True, cwd=HOME)
    assert usage.returncode == 1
    assert usage.stderr == "usage: harvest.py <telemetry.log> <alerts.txt>\n"
    missing = subprocess.run(
        [sys.executable, "harvest.py", str(td / "nope.log"), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert missing.returncode == 1
    assert missing.stderr == "bad telemetry: not found\n"
print("CH OK")